# Segment delay breakdown (MIT thesis §3.3)

For each signal-bounded segment and each daytime trip, decompose observed travel time (Eq 3.6):

`T_obs = T_ff + T_dwell + D_signal + D_congestion + Loss`

- **Segments** run between consecutive signals; segments that END at a near-side signal are ignored.
- **T_ff** (free-flow) = 5th-percentile travel time of nighttime trips (start 21:00–06:00).
- **T_dwell**: stops near a bus stop or near-side signal are treated as dwell and excluded from signal/congestion.
- **D_signal = uniform + overflow**, measured from stop durations: the stop closest to the downstream signal (within 100 m) is uniform, capped at the red-phase length (95th-pct of night stop-bar stops); additional upstream stops within 200 m connected by creeping (stop-slow-stop, no faster period) are overflow.
- **D_congestion**: residual (Eq 3.21) = max(0, T_obs − T_ff − T_dwell − D_signal). Loss is neglected.

All calculations live in `segment_delay.py`; tunable parameters (night window, percentiles, signal-stop area, faster-period speed) are module constants there.

In [ ]:
import geopandas as gpd
import pandas as pd
from IPython.display import display

from constants import CA_NAD83_Albers, CULVER_CITY_FEED_KEY, SERVICE_DATE, SHAPE_KEY_TO_SHAPE_ID_MAP
from _data_loaders import (
    get_culver_city_vehicle_positions,
    get_selected_shapes,
    get_traffic_signals,
    list_available_service_dates,
)
from segment_delay import run_segment_delay_analysis

## Configuration

In [ ]:
SHAPE_KEY = "105"
SHAPE_ID = SHAPE_KEY_TO_SHAPE_ID_MAP[SHAPE_KEY]
print(f"Shape: {SHAPE_KEY} -> {SHAPE_ID}")

## Load inputs

All trips for the shape across every service date, plus the shape geometry, signals, and stops (with hand-curated near-side flags). Loading all dates reads every per-date geoparquet, so this cell is the slow one.

In [ ]:
service_dates = list_available_service_dates()
vehicle_positions = pd.concat(
    [
        get_culver_city_vehicle_positions([SHAPE_KEY], service_date).assign(service_date=service_date)
        for service_date in service_dates
    ],
    ignore_index=True,
)
print(f"{len(vehicle_positions):,} positions over {len(service_dates)} dates")

shapes = get_selected_shapes(SERVICE_DATE, CULVER_CITY_FEED_KEY, [SHAPE_ID])
signals = get_traffic_signals()
stops = gpd.read_file(f"data/stops_{SHAPE_ID}.geojson").to_crs(CA_NAD83_Albers)

## Run the breakdown

In [ ]:
result = run_segment_delay_analysis(vehicle_positions, shapes, signals, stops, SHAPE_ID)
print(f"{len(result.segments)} analyzed segments (between signals, not ending at a near-side signal)")
result.segments.round(1)

## Trip numbers per segment

Daytime trips per segment: total, how many stopped at the signal, how many had overflow, how many had any congestion delay, plus the nighttime sample size behind each segment's baselines.

In [ ]:
result.trip_counts

## Raw travel times per segment

Descriptive statistics of observed daytime segment travel times (seconds), with the free-flow travel time (T_ff) alongside. The matrix below is one daytime trip per row, one segment per column.

In [ ]:
display(result.raw_travel_times.round(1))
display(result.travel_time_matrix.round(1))

## Delays per segment

Descriptive statistics (seconds) of each delay component over daytime trips: uniform, overflow, total signal, and congestion delay.

In [ ]:
for component in ["uniform_delay_s", "overflow_delay_s", "signal_delay_s", "congestion_delay_s"]:
    print(f"{component} by segment:")
    display(result.component_stats[component].round(1))

## Segment summary

One row per segment: geometry, free-flow time, red-phase cap, trip counts, and mean uniform / overflow / signal / congestion / dwell.

In [ ]:
result.segment_summary.round(1)

## Per trip-segment detail

Every daytime (trip, segment) with observed travel time, free-flow time, dwell, and the uniform / overflow / signal / congestion / total delay components.

In [ ]:
result.daytime_delays.round(1)

## Map of segments

Each analyzed segment is drawn along the shape in a distinct color so adjacent segments are easy to tell apart. Click a segment to see all of its summary values (free-flow time, red-phase cap, trip counts, mean delay components).

In [ ]:
import folium
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

from segment_delay import segment_geometries

segment_geometry = segment_geometries(shapes, SHAPE_ID, result.segments)[["geometry"]]
segment_map = gpd.GeoDataFrame(
    result.segment_summary.round(1).join(segment_geometry),
    geometry="geometry",
    crs=shapes.crs,
).reset_index().to_crs("EPSG:4326")

# Give each segment a distinct color (cycled in route order) so neighbors differ.
palette = [mcolors.to_hex(c) for c in plt.get_cmap("tab20").colors]
segment_map["color"] = [palette[i % len(palette)] for i in range(len(segment_map))]

popup_fields = [c for c in segment_map.columns if c not in ("geometry", "color")]

min_lon, min_lat, max_lon, max_lat = segment_map.total_bounds
segment_delay_map = folium.Map(
    location=[(min_lat + max_lat) / 2, (min_lon + max_lon) / 2],
    zoom_start=13,
    tiles="CartoDB positron",
)
folium.GeoJson(
    segment_map,
    style_function=lambda feature: {
        "color": feature["properties"]["color"],
        "weight": 5,
        "opacity": 0.85,
    },
    popup=folium.GeoJsonPopup(fields=popup_fields),
).add_to(segment_delay_map)
segment_delay_map

In [ ]:
example_detail = analyze_trip_segment(
    vehicle_positions, shapes, signals, stops, SHAPE_ID,
    "2026-02-13", 1376, "sig40_to_sig41"
)
plot_trip_segment(
    example_detail,
    title=f"{example.example}: {example.segment_id} — {example.service_date} trip {example.TRIP_KEY}",
)
plt.show()